# 1 · Build the cube

Set `AREA` and run. Reprojects every scene onto one common grid and stacks it
into a `(year, band, y, x)` cube, with a cloud/shadow layer from the QA bands.

Changes from the previous version:

* **Area switch** — `vatna` and `east` are configured side by side, so the same
  notebook builds either cube instead of needing hand edits.
* **`TARGET_RES = 10`** to match the re-acquired Sentinel-2 scenes. 30 m is what
  made the predictions look boxy.
* **Dates come from the filenames** (`*_YYYYMMDD.tif`), so the cloud mask uses
  each scene's own acquisition date — no separate date files needed, and it
  works for the automatically-selected east dates.
* Fixed `LANDSAT_MAX_YEAR`: 2016 is Sentinel-2, so the cutoff is **2015**.

In [1]:
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import xarray as xr
import rioxarray  # registers the .rio accessor
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
import matplotlib.pyplot as plt

import pystac_client
import odc.stac
try:
    import planetary_computer as pc
except ImportError:
    pc = None
    print("planetary-computer not installed -> Landsat cloud masks will be skipped")

### Configuration — pick the area here

In [2]:
PROJECT_ROOT = Path(r"D:\Users\b1120440\projects\morpheo\glacial_lake_pred")

AREA = "east"          # "vatna" or "east"

AREAS = {
    "vatna": {
        "scenes": Path(PROJECT_ROOT / "scenes/vatna"),
        "bbox": {"west": -16.5, "south": 63.97, "east": -16.1, "north": 64.2},
        "out": str(PROJECT_ROOT / "vatna_cube.zarr"),
    },
    "east": {
        "scenes": Path(PROJECT_ROOT / "scenes/east"),
        "bbox": {"west": -15.92166, "south": 64.26290,
                 "east": -15.25765, "north": 64.50511},
        "out": str(PROJECT_ROOT / "east_cube.zarr"),
    },
}

CFG = AREAS[AREA]
SCENES_DIR = CFG["scenes"]
BBOX = CFG["bbox"]
BBOX_LIST = [BBOX["west"], BBOX["south"], BBOX["east"], BBOX["north"]]
OUT_ZARR = CFG["out"]

TARGET_CRS = "EPSG:32627"
TARGET_RES = 10                     # must match the resolution used in notebook 0

# Export order written by notebook 0 - do not reorder.
BAND_NAMES = ["red", "green", "blue", "nir"]

LANDSAT_MAX_YEAR = 2017           # <=2015 Landsat, >=2016 Sentinel-2

PC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
ES_URL = "https://earth-search.aws.element84.com/v1"

print(f"AREA={AREA}  {SCENES_DIR}  res={TARGET_RES} m  -> {OUT_ZARR}")

AREA=east  D:\Users\b1120440\projects\morpheo\glacial_lake_pred\scenes\east  res=10 m  -> D:\Users\b1120440\projects\morpheo\glacial_lake_pred\east_cube.zarr


In [3]:
# ------------------------------------------------------------------
# One reference grid from the bbox, so every scene lands on it exactly
# ------------------------------------------------------------------
xmin, ymin, xmax, ymax = transform_bounds("EPSG:4326", TARGET_CRS, *BBOX_LIST)
xmin = np.floor(xmin / TARGET_RES) * TARGET_RES
ymin = np.floor(ymin / TARGET_RES) * TARGET_RES
xmax = np.ceil(xmax  / TARGET_RES) * TARGET_RES
ymax = np.ceil(ymax  / TARGET_RES) * TARGET_RES
width  = int((xmax - xmin) / TARGET_RES)
height = int((ymax - ymin) / TARGET_RES)
xs = xmin + TARGET_RES * (np.arange(width) + 0.5)
ys = ymax - TARGET_RES * (np.arange(height) + 0.5)

reference = (xr.DataArray(np.zeros((height, width), "float32"),
                          dims=("y", "x"), coords={"y": ys, "x": xs})
             .rio.write_crs(TARGET_CRS))
print(f"reference grid: {height} x {width} px @ {TARGET_RES} m "
      f"({height*width/1e6:.1f} Mpx per band-year)")

reference grid: 2965 x 3425 px @ 10 m (10.2 Mpx per band-year)


### Load & align every scene

Filenames are `<prefix>_YYYYMMDD.tif`, so both the year and the exact
acquisition date are read straight from the name.

In [4]:
def parse_name(stem: str):
    """-> (year, 'YYYY-MM-DD') from '<prefix>_YYYYMMDD'; date None if absent."""
    m8 = re.search(r"(19|20)\d{6}", stem)
    if m8:
        d = datetime.strptime(m8.group(0), "%Y%m%d")
        return d.year, d.strftime("%Y-%m-%d")
    m4 = re.search(r"(19|20)\d{2}", stem)
    return (int(m4.group(0)), None) if m4 else (None, None)


def load_scene(path: Path) -> xr.DataArray:
    da = rioxarray.open_rasterio(path, masked=True)      # (band, y, x), nodata -> NaN
    da = da.rio.reproject_match(reference, resampling=Resampling.bilinear)
    n = da.sizes["band"]
    if n != len(BAND_NAMES):
        print(f"  ! {path.name}: {n} bands but BAND_NAMES has {len(BAND_NAMES)}")
    return da.assign_coords(band=BAND_NAMES[:n])


records, date_of = {}, {}
for f in sorted(SCENES_DIR.glob("*.tif")):
    year, date = parse_name(f.stem)
    if year is None:
        print(f"skip {f.name}: no date in filename"); continue
    if year in records:
        print(f"  ! duplicate year {year} ({f.name}) - keeping the first"); continue
    records[year] = load_scene(f)
    date_of[year] = date
    print(f"{year}: {f.name}" + ("" if date else "   (no date -> no cloud mask)"))

assert records, f"no scenes in {SCENES_DIR} - check the path"
years = sorted(records)
print(f"\n{len(years)} scenes: {years[0]}-{years[-1]}")

1985: east_19850830.tif
1986: east_19860902.tif
1987: east_19870827.tif
1988: east_19880705.tif
1989: east_19890731.tif
1990: east_19900929.tif
1991: east_19910907.tif
1992: east_19920716.tif
1993: east_19930615.tif
1994: east_19940611.tif
1995: east_19950630.tif
1996: east_19960929.tif
1997: east_19970628.tif
1998: east_19980710.tif
1999: east_19990820.tif
2000: east_20000916.tif
2001: east_20010903.tif
2002: east_20020627.tif
2009: east_20090917.tif
2013: east_20130905.tif
2014: east_20140906.tif
2015: east_20150925.tif
2016: east_20160927.tif
2017: east_20170726.tif
2018: east_20180926.tif
2019: east_20190731.tif
2020: east_20200814.tif
2021: east_20210730.tif
2022: east_20220923.tif
2023: east_20230912.tif
2024: east_20240714.tif
2025: east_20250817.tif
2026: east_20260810.tif

33 scenes: 1985-2026


In [5]:
# ------------------------------------------------------------------
# Stack into one (year, band, y, x) cube
# ------------------------------------------------------------------
cube = (xr.concat([records[y] for y in years], dim="year")
          .assign_coords(year=years)
          .to_dataset(dim="band")
          .rio.write_crs(TARGET_CRS))
cube = cube.assign_coords(sensor=xr.DataArray(
    ["Landsat" if y <= LANDSAT_MAX_YEAR else "Sentinel-2" for y in years],
    dims="year", coords={"year": years}))

vmax = float(cube[BAND_NAMES].to_array().max())
print("cube:", dict(cube.sizes), "| bands:", list(cube.data_vars), "| max:", round(vmax, 3))
cube

cube: {'year': 33, 'y': 2965, 'x': 3425} | bands: ['red', 'green', 'blue', 'nir'] | max: 1.0


<xarray.Dataset> Size: 5GB
Dimensions:      (year: 33, y: 2965, x: 3425)
Coordinates:
  * year         (year) int64 264B 1985 1986 1987 1988 ... 2023 2024 2025 2026
    sensor       (year) <U10 1kB 'Landsat' 'Landsat' ... 'Sentinel-2'
  * y            (y) float64 24kB 7.166e+06 7.166e+06 ... 7.136e+06 7.136e+06
  * x            (x) float64 27kB 7.437e+05 7.437e+05 ... 7.779e+05 7.779e+05
    spatial_ref  int64 8B 0
Data variables:
    red          (year, y, x) float32 1GB 0.9051 0.9051 0.9051 ... 0.0838 0.0773
    green        (year, y, x) float32 1GB 0.9402 0.9402 0.9402 ... 0.0728 0.0695
    blue         (year, y, x) float32 1GB 1.0 1.0 1.0 ... 0.0617 0.0635 0.0538
    nir          (year, y, x) float32 1GB 0.8165 0.8165 0.8165 ... 0.1218 0.1142
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0
    _FillValue:     nan

### Cloud / shadow layer

Snow is deliberately **not** flagged — over a glacier it must stay valid.

In [6]:
_es = pystac_client.Client.open(ES_URL)
_pc = pystac_client.Client.open(PC_URL, modifier=pc.sign_inplace) if pc else None


def _load_qa(cat, collection, band, date):
    day = f"{date}T00:00:00Z/{date}T23:59:59Z"
    items = list(cat.search(collections=[collection], bbox=BBOX_LIST, datetime=day).items())
    if not items:
        return None
    da = odc.stac.load(items, bands=[band], crs=TARGET_CRS, resolution=TARGET_RES,
                       bbox=BBOX_LIST, groupby="solar_day", resampling="nearest",
                       chunks={}).isel(time=0)[band].compute()
    return da.rio.reproject_match(reference, resampling=Resampling.nearest)


def cloud_for(year):
    """True where cloud / cloud-shadow / cirrus. Snow is NOT flagged."""
    date = date_of.get(year)
    if date is None:
        return None
    if year <= LANDSAT_MAX_YEAR:
        if _pc is None:
            return None
        qa = _load_qa(_pc, "landsat-c2-l2", "qa_pixel", date)
        if qa is None:
            return None
        bits = (1 << 1) | (1 << 2) | (1 << 3) | (1 << 4)   # dilated|cirrus|cloud|shadow
        return (qa.astype("uint16").values & bits) != 0
    scl = _load_qa(_es, "sentinel-2-l2a", "scl", date)
    if scl is None:
        return None
    return np.isin(scl.values, [3, 8, 9, 10])              # shadow|cloud|cloud|cirrus


cloud_arr = []
for y in years:
    try:
        cl = cloud_for(y)
    except Exception as e:
        print(f"{y}: cloud fetch failed ({e}); no mask"); cl = None
    if cl is None:
        cl = np.zeros((height, width), bool)
    cloud_arr.append(cl)
    print(f"{y}: cloud/shadow {cl.mean()*100:5.1f}%")

cube["cloud"] = xr.DataArray(np.stack(cloud_arr), dims=("year", "y", "x"),
                             coords={"year": years, "y": cube.y, "x": cube.x})

1985: cloud/shadow   6.6%
1986: cloud/shadow  12.1%
1987: cloud/shadow   1.6%
1988: cloud/shadow   0.7%
1989: cloud/shadow  13.1%
1990: cloud/shadow  12.7%
1991: cloud/shadow   1.1%
1992: cloud/shadow   0.0%
1993: cloud/shadow   0.0%
1994: cloud/shadow   0.5%
1995: cloud/shadow   0.7%
1996: cloud/shadow   9.7%
1997: cloud/shadow   0.9%
1998: cloud/shadow  12.4%
1999: cloud/shadow  23.6%
2000: cloud/shadow   3.9%
2001: cloud/shadow  15.5%
2002: cloud/shadow   1.0%
2009: cloud/shadow   3.6%
2013: cloud/shadow   0.2%
2014: cloud/shadow   2.3%
2015: cloud/shadow   6.1%
2016: cloud/shadow  17.9%
2017: cloud/shadow   0.0%
2018: cloud/shadow   1.3%
2019: cloud/shadow   0.7%
2020: cloud/shadow   0.7%
2021: cloud/shadow   0.0%
2022: cloud/shadow   0.7%
2023: cloud/shadow   2.7%
2024: cloud/shadow   0.1%
2025: cloud/shadow   4.5%
2026: cloud/shadow   0.0%


### Check

In [12]:
# def rgb(ds_year, p=(2, 98)):
#     a = np.dstack([ds_year.red, ds_year.green, ds_year.blue])
#     lo, hi = np.nanpercentile(a, p)
#     return np.clip((a - lo) / (hi - lo + 1e-9), 0, 1)

# show = years[0:] if len(years) >= 1 else years
# fig, ax = plt.subplots(1, len(show), figsize=(6*len(show), 6))
# for a, yr in zip(np.atleast_1d(ax), show):
#     a.imshow(rgb(cube.sel(year=yr))); a.axis("off")
#     a.set_title(f"{yr} · {cube.sensor.sel(year=yr).values}")
# plt.tight_layout(); plt.show()

In [8]:
cube = cube.chunk({"year": 1, "y": 1024, "x": 1024})
cube.to_zarr(OUT_ZARR, mode="w")
print(f"wrote {OUT_ZARR} with bands: {list(cube.data_vars)}")

C:\Users\b1120440\AppData\Local\miniconda3\envs\eocube\Lib\site-packages\zarr\core\dtype\npy\string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=10, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
C:\Users\b1120440\AppData\Local\miniconda3\envs\eocube\Lib\site-packages\zarr\api\asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


wrote D:\Users\b1120440\projects\morpheo\glacial_lake_pred\east_cube.zarr with bands: ['red', 'green', 'blue', 'nir', 'cloud']


### Verify the band mapping

Both areas go through the same `to_reflectance`, so their medians should be
broadly comparable. If one is ~2x the other, something diverged upstream.

In [9]:
import xarray as xr, numpy as np, pandas as pd
chk = xr.open_zarr(OUT_ZARR)
rows = []
for yr in [int(y) for y in chk.year.values]:
    med = {}
    for v in ["red","green","blue","nir"]:
        x = chk[v].sel(year=yr).values
        x = x[(x>0) & np.isfinite(x)]
        med[v] = float(np.median(x)) if x.size else np.nan
    rows.append({"year": yr, "sensor": str(chk.sensor.sel(year=yr).values),
                 **{k: round(v,3) for k,v in med.items()},
                 "blue/nir": round(med["blue"]/med["nir"], 2),
                 "cloud%": round(float(chk.cloud.sel(year=yr).mean())*100, 1)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nflagged (hazy):", df[df["blue/nir"] > 0.8].year.tolist())

 year     sensor   red  green  blue   nir  blue/nir  cloud%
 1985    Landsat 0.270  0.277 0.263 0.308      0.85     6.6
 1986    Landsat 0.123  0.118 0.096 0.222      0.43    12.1
 1987    Landsat 0.115  0.114 0.090 0.199      0.45     1.6
 1988    Landsat 0.131  0.126 0.101 0.215      0.47     0.7
 1989    Landsat 0.129  0.127 0.109 0.224      0.49    13.1
 1990    Landsat 0.131  0.129 0.104 0.231      0.45    12.7
 1991    Landsat 0.112  0.109 0.085 0.179      0.48     1.1
 1992    Landsat 0.594  0.584 0.456 0.562      0.81     0.0
 1993    Landsat 0.827  0.866 1.000 0.667      1.50     0.0
 1994    Landsat 0.465  0.488 0.446 0.381      1.17     0.5
 1995    Landsat 0.166  0.163 0.164 0.240      0.68     0.7
 1996    Landsat 0.117  0.116 0.089 0.219      0.41     9.7
 1997    Landsat 0.143  0.137 0.116 0.222      0.52     0.9
 1998    Landsat 0.146  0.150 0.129 0.260      0.50    12.4
 1999    Landsat 0.162  0.160 0.147 0.235      0.62    23.6
 2000    Landsat 0.109  0.101 0.081 0.20